# Sprint 2 — U-Net Baseline Training

**Flow:** prepare_data.py -> Dataset -> U-Net (ResNet50) -> MLflow

Run `uv run python scripts/prepare_data.py` before executing this notebook.

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

## 1. Verify patch dataset

In [ ]:
from pasture.data.dataset import PastureDataset
import matplotlib.pyplot as plt
import numpy as np

patch_dir = project_root / 'data' / 'patches'
train_ds = PastureDataset(patch_dir / 'train', augment=False)
val_ds   = PastureDataset(patch_dir / 'val',   augment=False)
print(f'Train patches: {len(train_ds)}')
print(f'Val   patches: {len(val_ds)}')

img, msk = train_ds[0]
print(f'Image shape : {img.shape}  dtype: {img.dtype}')
print(f'Mask  shape : {msk.shape}  dtype: {msk.dtype}')
print(f'Mask classes: {msk.unique().tolist()}')

In [ ]:
# Visualise a sample patch
LABEL_COLORS = ['#aaaaaa', '#2d6a2d', '#d4e157', '#c4a35a', '#4fc3f7']
LABEL_NAMES  = ['ignored', 'healthy', 'stressed', 'bare', 'water']
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample training patches (true colour | mask)', fontsize=12)

from pasture.data.dataset import BAND_MEAN, BAND_STD

for col in range(4):
    img, msk = train_ds[col * len(train_ds) // 4]
    # Denormalise RGB for display
    rgb = img[:3].numpy() * BAND_STD[:3, None, None] + BAND_MEAN[:3, None, None]
    rgb = np.transpose(np.clip(rgb / 0.25, 0, 1), (1, 2, 0))

    axes[0, col].imshow(rgb)
    axes[0, col].set_title(f'Patch {col}')
    axes[0, col].axis('off')

    cmap = mcolors.ListedColormap(LABEL_COLORS)
    norm = mcolors.BoundaryNorm([-0.5+i for i in range(6)], cmap.N)
    axes[1, col].imshow(msk.numpy(), cmap=cmap, norm=norm)
    axes[1, col].axis('off')

legend = [Patch(facecolor=c, label=n) for c, n in zip(LABEL_COLORS[1:], LABEL_NAMES[1:])]
fig.legend(handles=legend, loc='lower center', ncol=4, fontsize=9)
plt.tight_layout()
plt.show()

## 2. Train U-Net baseline

In [ ]:
# Runs training and logs everything to mlruns/
# Open MLflow UI after: uv run mlflow ui
from pasture.training.train import train

train(
    encoder='resnet50',
    epochs=30,
    batch_size=16,
    lr=1e-4,
    patch_dir=str(project_root / 'data' / 'patches'),
    ckpt_dir=str(project_root / 'data' / 'checkpoints'),
)

## 3. Inspect predictions

In [ ]:
import torch
from pasture.models.unet import build_unet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load(project_root / 'data' / 'checkpoints' / 'best.pt',
                  map_location=device)
model = build_unet(encoder=ckpt['encoder']).to(device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Loaded checkpoint  epoch={ckpt["epoch"]}  mIoU={ckpt["miou"]:.4f}')

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Val set: True Colour | Ground Truth | Prediction', fontsize=12)

cmap = mcolors.ListedColormap(LABEL_COLORS)
norm = mcolors.BoundaryNorm([-0.5+i for i in range(6)], cmap.N)

with torch.no_grad():
    for col in range(4):
        img, msk = val_ds[col * len(val_ds) // 4]
        logits = model(img.unsqueeze(0).to(device))
        pred   = logits.argmax(1).squeeze().cpu().numpy()

        rgb = img[:3].numpy() * BAND_STD[:3, None, None] + BAND_MEAN[:3, None, None]
        rgb = np.transpose(np.clip(rgb / 0.25, 0, 1), (1, 2, 0))

        axes[0, col].imshow(rgb)
        axes[0, col].set_title(f'Patch {col}')
        axes[0, col].axis('off')

        axes[1, col].imshow(msk.numpy(), cmap=cmap, norm=norm)
        axes[1, col].set_title('Ground truth')
        axes[1, col].axis('off')

        axes[2, col].imshow(pred, cmap=cmap, norm=norm)
        axes[2, col].set_title('Prediction')
        axes[2, col].axis('off')

legend = [Patch(facecolor=c, label=n) for c, n in zip(LABEL_COLORS[1:], LABEL_NAMES[1:])]
fig.legend(handles=legend, loc='lower center', ncol=4, fontsize=9)
plt.tight_layout()
out = project_root / 'data' / 'val_predictions.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {out}')